# What a CRISM band is

A CRISM observation arrives as a cube of numbers with no wavelengths anywhere in it. The
label says the file holds 55 bands and says nothing about what any of those 55 bands
looked at. Band 0 of one observation and band 0 of the next need not be the same colour
of light.

## How CRISM creates data

CRISM scans Mars one thin line at a time, just like a document scanner. As the spacecraft flies forward, light from a single strip of ground enters the camera. Inside, a glass grating splits that light into a rainbow across a rectangular sensor:

* **Columns (left to right):** Each column is a specific spot on the ground.
* **Rows (top to bottom):** Each row is a specific color of light.

Stacking these snapshots over time creates a 3D data file made of three parts:

* **Lines (Time & Distance):** The forward movement of the spacecraft along its flight path over time.
* **Samples (Position):** The location of spots going across the scanned line.
* **Bands (Sensor Rows):** The individual rows on the sensor. A row's assigned color depends on how the internal optics split the light.

CRISM uses two separate sensors that save into separate files:

* **`S` sensor:** Captures visible light.
* **`L` sensor:** Captures infrared light.

Because each sensor records a different set and number of color rows, every single scan generates two 3D data files of different sizes.

## Setup

In [ ]:
"""Bring one observation down and name what to look at inside it."""

import numpy as np

from preprocessing.crism import reading
from preprocessing.crism.clean import clean
from preprocessing.crism.cleaning import atmospheric, destriping, masking
from preprocessing.crism.fetching import bands_calibration, download
from preprocessing.crism.loaders import wavelengths
from preprocessing.crism.loaders.utils import locations, naming, pds

OBSERVATION = "msp00006994_05_if214_trr3"

LINE, SAMPLE = 1000, 32

for detector, label in download.fetch(OBSERVATION).items():
    print(detector, label.name)

In [ ]:
"""Two printers, so every table below is laid out the same way."""


def edges(size, keep=3):
    """Return the first and last few indices of an axis.

    Args:
        size: How long the axis is.
        keep: How many indices to take from each end.

    Returns:
        The indices, first end then last.
    """
    return list(range(keep)) + list(range(size - keep, size))


def block(grid, rows, cols, digits=2):
    """Print part of a two dimensional grid as a plain table.

    Args:
        grid: The grid to read from.
        rows: Which row indices to print, in the order to print them.
        cols: Which column indices to print, in the order to print them.
        digits: How many decimals to show each value with.

    Returns:
        None.
    """
    print("      " + "".join(f"{col:>10}" for col in cols))
    for row in rows:
        cells = "".join(f"{grid[row, col]:>10.{digits}f}" for col in cols)
        print(f"{row:>6}" + cells)

## The TRDR

The label is the only description the file carries of its own shape. It gives the three
axis lengths, the order the values were written in, and the width of one number. It does
not give a single wavelength.

`PIXEL_AVERAGING_WIDTH` is the binning: 10 detector columns per sample, which is where 64
samples comes from.

In [ ]:
"""What each detector's label says about the shape of its cube."""

head = f"{'det':>4}{'lines':>8}{'samples':>9}{'bands':>7}{'bin':>5}"
print(f"{head}  {'order':<19}unit")
for name in naming.DETECTORS:
    image = locations.files(OBSERVATION, name)[".img"]
    label = pds.load_label(image.with_suffix(".lbl"))
    lines, samples, bands, stored, dtype = pds.load_layout(label)
    row = f"{name:>4}{lines:>8}{samples:>9}{bands:>7}"
    print(f"{row}{label['PIXEL_AVERAGING_WIDTH']:>5}  {stored:<19}{label['UNIT']}")

### One band of the raw cube

The file writes `65535.0` there, which is a flag meaning the detector was never calibrated at that column.
Left as a number it is roughly two hundred thousand times a real value, so any average that includes it is ruined.

In [ ]:
"""The infrared cube exactly as the file writes it, at one band."""

image = locations.files(OBSERVATION, "l")[".img"]
label = pds.load_label(image.with_suffix(".lbl"))
raw = pds.build_cube(image, label)

print(f"cube {raw.shape}, band fixed 10, lines rows, samples columns\n")
block(raw[:, :, 10], edges(raw.shape[0]), edges(raw.shape[1]), digits=3)

### One pixel of the raw cube

Fixing a line and a sample and reading across the bands gives a spectrum. This is what the
file offers: a value per band index, and no way to know what band index means.

Band 0 is `65535.0` down the whole cube. That row of the detector was downlinked but never
calibrated, so the entire band is a flag.

In [ ]:
"""The spectrum the file gives at one pixel, indexed only by band number."""

print(f"line {LINE}, sample {SAMPLE}, {raw.shape[2]} bands")
print("".join(f"{band:>10}" for band in edges(raw.shape[2], 5)))
print("".join(f"{raw[LINE, SAMPLE, band]:>10.3f}" for band in edges(raw.shape[2], 5)))

## The CDR
A Calibration Data Record (CDR) is a pre-launch reference file that describes the instrument rather than a specific observation.

The WA record is the wavelength CDR. It maps every detector column and row to its center wavelength in nanometers. It is a 2D table because the instrument projects light in a slight curve across the sensor. This optical effect, called spectral smile, causes the exact wavelength of a single row to shift from left to right.

Observation labels state which WA record to use to look up these wavelengths.

In [ ]:
"""Which wavelength record each detector of this observation names."""

records = {}
for name in naming.DETECTORS:
    label = pds.load_label(locations.files(OBSERVATION, name)[".lbl"])
    records[name] = locations.wavelength_file(naming.wavelength(label))[".img"]
    print(f"{name}  bands {label['BANDS']:>3}  {label[naming.WAVELENGTH_KEY]}")
    print(f"   -> {records[name].name}")

### The record as bytes, then as the loader reads it

The `.img` of a CDR is built the same way an observation is: a plain grid of 32 bit floats
with a label beside it. It holds one line, 64 columns and one value per band, and it writes
the same `65535` flag where nothing was ever calibrated.

The first table is the file untouched. The second is what `wavelengths.load` returns, which
is the same grid with the flag turned into `NaN` so that averaging it is impossible rather
than merely wrong.

In [ ]:
"""The infrared record, as written and as loaded."""

bands = raw.shape[2]
stored = np.fromfile(records["l"], dtype="<f4", count=64 * bands)
stored = stored.reshape(bands, 64).T
loaded = wavelengths.load(records["l"])

print("as written, columns down, bands across")
block(stored, edges(64), edges(bands), digits=1)
print()
print("as loaded")
block(loaded, edges(64), edges(bands), digits=1)

In this way, comparing the TRDR and the CDR permits to define the smile error and the exact wavelength of each band in a specific observation.

In [ ]:
"""How far one band drifts in wavelength across the swath."""

for name, record in records.items():
    table = wavelengths.load(record)
    print(f"{name}, {table.shape[1]} bands")
    print(f"{'band':>6}{'min (nm)':>12}{'max (nm)':>12}{'spread':>10}")
    for band in edges(table.shape[1]):
        column = table[:, band]
        if np.isnan(column).all():
            print(f"{band:>6}{'never calibrated':>34}")
            continue
        low, high = np.nanmin(column), np.nanmax(column)
        print(f"{band:>6}{low:>12.2f}{high:>12.2f}{high - low:>10.2f}")
    print()

## Putting the two together

`bands_calibration.calibrate` does three things and nothing else. It takes the record it
is handed, without opening anything itself. It reverses the cube and the record together
when the record runs long wavelength first, so that every cube leaves in the same
direction. And it writes `NaN` over the columns and the bands the record never calibrated,
so the flag never survives as a number.

The shape does not change. A dead band stays a band, it just stops pretending to be data.

In [ ]:
"""The same pixel before and after, and what each band is centred on."""

ordered, table = bands_calibration.calibrate(raw, wavelengths.load(records["l"]))
centres = bands_calibration.centres(table)

print(f"{'band':>6}{'stored':>12}{'wavelength':>13}{'calibrated':>13}")
for band in edges(ordered.shape[2], 5):
    print(
        f"{band:>6}{raw[LINE, SAMPLE, band]:>12.3f}"
        f"{centres[band]:>13.1f}{ordered[LINE, SAMPLE, band]:>13.3f}"
    )

In [ ]:
"""The cube after calibration, at the band the raw slice was printed at."""

print(f"cube {ordered.shape}, band 44, lines down, samples across")
block(ordered[:, :, 44], edges(ordered.shape[0]), edges(ordered.shape[1]), digits=3)

## What this leaves the rest of the pipeline

A calibrated cube is still lines by samples by bands, and a band is still an index. What
has changed is that the index now means the same thing in every observation of a given
configuration: the bands ascend in wavelength, and the wavelengths themselves are carried
beside the cube rather than assumed.

Two things are deliberately not done here and are worth stating.

**Nothing is resampled.** The wavelength of a band still depends on the sample it is read
at, because the smile is real and is up to 16 nm on the infrared detector. Comparing one
sample against another at a fixed band index compares slightly different colours. The
wavelengths are kept as a full table for exactly that reason, so any later step can see the
drift instead of averaging it away.

**Dead columns and bands are kept in place.** They are `NaN`, not removed. Dropping them
would change the shape of the cube per configuration and break the correspondence with the
geometry backplanes, which are on the same grid.

**Nothing is guessed from the band count.** The label names one record outright, that one is
brought down beside the observation, and it is the only one read.

## What is left that is not a measurement

Ordering the bands does not make every number in the cube real. Three kinds of cell are
still not readings, and `cleaning.masking.bad_pixels` refuses all three.

**Dead columns and bands.** What the wavelength file never named, already `NaN` above.

**The sensor edges.** At the extreme ends of the detector almost no light lands, so the
reading is mostly noise divided by a tiny calibration number, which can come out any size.
These bands are bad by construction rather than by accident, so the whole band is dropped
rather than tested value by value. The window kept is 1020 to 2650 nm on the infrared
detector and 400 to 1060 nm on the visible one.

**Scattered values.** Anything left over that a brightness cannot be. A brightness is the
fraction of the arriving sunlight that came back up, so it cannot exceed 1 and it cannot
really be negative. The floor is set a little under zero rather than at it, because a
reading of very dark ground sits on the noise floor and lands just below zero without
being wrong. The range kept is `-0.05` to `1`.

Nothing is deleted, because the cube has to stay a rectangle and a hole would spread: any
average taken down a column that met one would return a hole. Every refused cell is
replaced by the mean of what is kept, and the `Mask` beside the cube remembers where that
happened, so nothing later mistakes a stand-in for a measurement.

In [ ]:
"""How much of each cube is edge, and how much is scattered."""

for name, detector in clean(OBSERVATION).detectors.items():
    mask = detector.mask
    columns, bands = mask.kept
    total = detector.cube[:, ~mask.columns][:, :, ~mask.bands].size
    print(f"{name}  {detector.cube.shape} -> {columns} columns by {bands} bands kept")
    print(
        f"   dead columns        : {int(mask.columns.sum()):>3} of {mask.columns.size}"
    )
    named = mask.bands & ~mask.edges & ~mask.atmospheric
    print(f"   bands never named    : {int(named.sum()):>3}")
    print(f"   bands dropped as edge: {int(mask.edges.sum()):>3}")
    print(f"   bands dropped as air : {int(mask.atmospheric.sum()):>3}")
    print(f"   scattered values    : {int(mask.scattered.sum()):>3} of {total:,} kept")
    print(f"   unusable pixels     : {mask.pixels.mean():>7.2%}")
    print(f"   filled with         : {mask.fill:.4f}")

## Cells that read off in every line

Every line of the scan is read through the same grid of detector cells. So a cell that
reads a little high is a little high for all 2700 lines, and paints a stripe down the whole
image, one band deep and one column wide. A stripe is long and straight and survives
averaging, so it looks like a ridge or a compositional boundary while being glass.

To find one, average a column down the whole scan. Those 2700 readings saw 2700 different
patches of ground, so the ground averages away and what is left is the instrument.

`cleaning.destriping.remove_spike_columns` is `crism_ml`'s `remove_spikes_column`: it
compares each band of that averaged spectrum to the median of its wavelength neighbours,
and calls a band a spike when it sits more than a set number of standard deviations above
the mean departure of its own column.

Only the threshold is changed, and it is one per detector. `crism_ml` uses 5 over 248
hyperspectral channels. A survey scan carries 18 to 55, where a real absorption is often
one band wide and looks exactly like a spike: at 5 the test deletes the 1430 nm and 2010 nm
absorptions on the infrared detector and nothing else. Each threshold is set above the
highest a real absorption was measured to reach on its own detector, 7.07 for `l` and 4.59
for `s`.

In [ ]:
"""What was levelled, how far, and what was left alone."""

for name in naming.DETECTORS:
    part = reading.read(OBSERVATION).detectors[name]
    before, mask = masking.bad_pixels(part.cube, part.wavelengths, name)
    before, mask = atmospheric.remove_atmospheric_bands(
        before, mask, part.wavelengths, name
    )
    after, mask = destriping.remove_spike_columns(before, mask, part.wavelengths, name)
    centre = bands_calibration.centres(part.wavelengths)
    columns, bands = mask.stripes.nonzero()

    print(f"{name}  {int(mask.stripes.sum())} of {mask.stripes.size} cells levelled")
    for column, band in list(zip(columns, bands))[:6]:
        moved = after[:, column, band].mean() - before[:, column, band].mean()
        print(
            f"   column {column:>2} band {band:>2} at {centre[band]:>6.0f} nm"
            f"   moved {moved:>+9.5f} I/F"
        )
    print(f"   nothing else moved by more than {np.abs(after - before).max():.5f} I/F")
    kept = int((~mask.bands).sum())
    print(f"   bands touched: {len(set(bands.tolist()))} of {kept} kept")